# 01 — Jouvence data model and scientific use cases

Jouvence separates **entities** (`nodes/`), deduplicated biological **assertions** (`edges/`), source-specific **support/provenance** (`evidence/`), and node/edge **features** (`features/`). This separation matters: a graph edge states what the KG represents, while evidence records why that assertion is present and with what source context.

Useful questions include target–disease neighborhood inspection, provenance-aware hypothesis generation, representation retrieval, and leakage-controlled link prediction. None of these outputs independently proves causality, clinical efficacy, or mechanism.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


In [ ]:
from manage_db.kg_schema import NODE_TYPES, RELATIONS
from manage_db.kg_evidence import EVIDENCE_PARQUET_COLUMNS
import pandas as pd

node_types = pd.DataFrame([{
    "node_type": key.value,
    "primary_ontology": value.primary_ontology,
    "example_id": value.example_id,
} for key, value in NODE_TYPES.items()])
relations = pd.DataFrame([{
    "relation": rel.name,
    "source_type": rel.source.value,
    "target_type": rel.target.value,
    "kind": rel.kind.value,
    "status": rel.status.value,
} for rel in RELATIONS])
print(f"Declared node types: {len(node_types)}; declared relations: {len(relations)}")
display(node_types.head(10))
display(relations.head(10))


In [ ]:
catalog = parquet_catalog(KG_ROOT, billing_project=BILLING_PROJECT)
display(catalog[["layer", "path", "rows", "row_groups"]])
print("Fixture rows are examples for executable documentation, not release cardinalities.")


## Interpretation boundary

- Multiple source records can support one assertion; evidence count is not automatically evidence strength.
- Missing evidence in a bounded sample or currently partial LaminDB mirror does not prove absence from canonical Parquet.
- Provenance nodes and clinical-trial metadata are not default message-passing topology.
- Public embedding publication is separately gated by model identity, license, schema-compaction, and immutable-release checks.
